In [1]:
import os
import pandas as pd
import numpy as np
import boto3
from tqdm import tqdm
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
# Suppress PerformanceWarning
warnings.filterwarnings('ignore')
import sklearn.metrics as skm
import seaborn as sns
import matplotlib.pyplot as plt
import sklearn.metrics as skm
from sklearn.linear_model import LinearRegression

try:
    import catboost as cb
except:
    ! pip install catboost

### Functions

In [2]:
# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # download file
    boto3.client('s3').download_file(str_project, str_bucket_path, str_local_path)

### Constants

In [3]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')
str_dirname_output = './output'
str_variant = 'noPTImodel7'

Project: 20231010-gen-xii
Task: ad_hoc


### Output directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Load in data

In [5]:
list_str_df = [
    'train',
    'valid',
    'test',
]
list_df = []
for str_df in tqdm(list_str_df):
    str_filename = f'df_{str_df}_raw.gzip'
    str_uri = f's3://{str_project}/03_pricing_lgd/01_data_prep/03_train_valid_test_split/{str_filename}'
    df = pd.read_parquet(str_uri)
    df['data_set'] = str_df
    list_df.append(df)
df = pd.concat(list_df)
list_cols = [
    'decision_dte__base',
    'open_dte__base',
    'acct_typ_cde__base',
    'terms_freq_cde__base',
    'dtmstampcreation__base',
    'dtmapproved__base',
    'dtmdeclined__base',
    'observationdate__base',
    'analyticsmatchkey__base',
    'booked__base',
    'curr_bal_amt__base',
    'affil_rem_cde__base',
    'cmplnc_rem_cde__base',
    'gnrc_rem_cde__base',
    'rte_rem_cde__base',
    'acct_rte_prfl_cde__base',
    'payt_pttrn_txt__base',
    'gnrcrem_frsrpt_dte__base',
    'obs_arch__base',
    'vtg4__base',
    'performance__base',
    'creditasofdate__tu',
    'credit_as_of_date__tu',
    'strname__tu',
    'state__tu',
]
df.drop(list_cols, axis=1, inplace=True)
# show
df

100%|██████████| 3/3 [00:03<00:00,  1.12s/it]


,firstname__tu,middlename__tu,lastname__tu,address1__tu,city__tu,zip5__tu,zip4__tu,ssn__tu,dateofbirth__tu,dtmstampcreation__tu,...,payment__app,dti__app,pti__app,bitservicecontract__app,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app,target,data_set
8202,NaN,NaN,NaN,NaN,REYNOLDSBURG,NaN,NaN,NaN,NaN,NaN,...,398.83,0.229354,0.052130,1,1.106635,auto,0,2012-06-18 09:31:54.393,-0.097929,train
8203,NaN,NaN,NaN,NaN,REYNOLDSBURG,NaN,NaN,NaN,NaN,NaN,...,398.83,0.229354,0.052130,1,1.106635,auto,0,2012-06-18 09:31:54.393,-0.097929,train
8204,NaN,NaN,NaN,NaN,ENGLEWOOD,NaN,NaN,NaN,NaN,NaN,...,420.66,0.470570,0.153369,1,1.254334,auto,1,2012-03-06 16:36:21.623,0.464463,train
8205,NaN,NaN,NaN,NaN,ENGLEWOOD,NaN,NaN,NaN,NaN,NaN,...,420.66,0.470570,0.153369,1,1.254334,auto,1,2012-03-06 16:36:21.623,0.464463,train
8206,NaN,NaN,NaN,NaN,RICHARDSON,NaN,NaN,NaN,NaN,NaN,...,440.04,0.405641,0.050115,0,1.249982,auto,0,2009-10-15 16:06:40.767,0.543609,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1595,NaN,NaN,NaN,NaN,None,19151.0,NaN,NaN,NaN,20191025.0,...,471.42,0.319547,0.119549,0,1.144597,auto,0,2016-11-11 10:56:15.140,0.175335,test
2335,NaN,NaN,NaN,NaN,None,72112.0,NaN,NaN,NaN,20191025.0,...,592.62,0.493973,0.119966,0,1.333843,auto,0,2003-07-08 14:32:29.097,0.317968,test
5274,NaN,NaN,NaN,NaN,None,77050.0,NaN,NaN,NaN,20191026.0,...,407.00,0.424233,0.117298,0,1.136382,auto,0,2013-09-10 16:40:45.693,0.278606,test
6464,NaN,NaN,NaN,NaN,None,21798.0,NaN,NaN,NaN,20191026.0,...,400.83,0.320787,0.120292,1,1.076209,auto,1,2010-11-12 15:52:36.113,0.660134,test


### Preprocess

In [6]:
# preprocess
list_str_filename = [
    'preprocessing.py',
    'cls_model_preprocessing.pkl',
]
for str_filename in tqdm(list_str_filename):
    # download
    str_bucket_path = f'01_ad/02_model/{str_variant}/00_preprocessing/01_create_preprocessor/{str_filename}'
    str_local_path = f'./{str_filename}'
    download_from_s3(
        str_local_path=str_local_path, 
        str_bucket_path=str_bucket_path, 
        str_project=str_project,
    )
# import
cls_model_preprocessing = pickle.load(open(str_local_path, 'rb'))
# rm
os.remove(str_local_path)

# preprocess
list_target = list(df['target'])
list_cols = [col for col in df.columns if col != 'target']
df = cls_model_preprocessing.transform(df[list_cols])
df['target'] =  list_target

# rm
os.remove('./preprocessing.py')

# show
df

100%|██████████| 2/2 [00:00<00:00,  5.85it/s]


NaN Replacer: 1.2976 sec.


100%|██████████| 3/3 [00:00<00:00, 77.13it/s]

Set strings: 0.040862 sec.


Boolean Replacer: 1.5081 sec.


...


100%|██████████| 2457/2457 [00:02<00:00, 1211.93it/s]


Replace inf and -inf with NaN: 2.4423 sec.
Imputer: 1.0836 sec.
Map term: 0.029534 sec.
Map PTI: 0.031923 sec.


100%|██████████| 9/9 [00:00<00:00, 1455.18it/s]

Round values: 0.0099061 sec.
Preprocessing Model: 13.17 sec.


,firstname__tu,middlename__tu,lastname__tu,address1__tu,city__tu,zip5__tu,zip4__tu,ssn__tu,dateofbirth__tu,dtmstampcreation__tu,...,data_set,year,factor,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-payment_to_income,ENG-loan_to_value,ENG-vehicle_age,ENG-dealership_age,target
8202,0.0,0.0,0.0,0.0,reynoldsburg,0.0,0.0,0.0,0.0,0.0,...,train,2013,1.256262,10,4,0.09,1.370370,4.0,1.284932,-0.097929
8203,0.0,0.0,0.0,0.0,reynoldsburg,0.0,0.0,0.0,0.0,0.0,...,train,2013,1.256262,10,4,0.09,1.370370,4.0,1.284932,-0.097929
8204,0.0,0.0,0.0,0.0,englewood,0.0,0.0,0.0,0.0,0.0,...,train,2013,1.256262,10,4,0.15,1.586207,2.0,1.569863,0.464463
8205,0.0,0.0,0.0,0.0,englewood,0.0,0.0,0.0,0.0,0.0,...,train,2013,1.256262,10,4,0.15,1.586207,2.0,1.569863,0.464463
8206,0.0,0.0,0.0,0.0,richardson,0.0,0.0,0.0,0.0,0.0,...,train,2013,1.256262,10,4,0.03,1.342857,1.0,3.961644,0.543609
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1595,0.0,0.0,0.0,0.0,nan,19151.0,0.0,0.0,0.0,20191025.0,...,test,2019,1.144717,10,4,0.09,1.236842,1.0,2.953425,0.175335
2335,0.0,0.0,0.0,0.0,nan,72112.0,0.0,0.0,0.0,20191025.0,...,test,2019,1.144717,10,4,0.12,1.500000,3.0,16.306849,0.317968
5274,0.0,0.0,0.0,0.0,nan,77050.0,0.0,0.0,0.0,20191026.0,...,test,2019,1.144717,10,4,0.09,1.290323,3.0,6.126027,0.278606
6464,0.0,0.0,0.0,0.0,nan,21798.0,0.0,0.0,0.0,20191026.0,...,test,2019,1.144717,10,4,0.09,1.233333,3.0,8.956164,0.660134


### Get PD predictions

In [7]:
# get gen 12 PD model
str_filename = 'final_model.pkl'
str_bucket_path = f'02_pricing_pd/02_model/{str_variant}/03_final_model/{str_filename}'
str_local_path = f'{str_dirname_output}/{str_filename}'
download_from_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)
cls_model_inference = pickle.load(open(str_local_path, 'rb'))['model_inference']
os.remove(str_local_path)
# predict
list_cols_model = list(cls_model_inference.feature_names_)
df['pd'] = cls_model_inference.predict_proba(df[list_cols_model])[:,1]
# subset
list_cols = [
    'data_set',
    'pd',
    'target',
]
df = df[list_cols].copy()
# show
df

,data_set,pd,target
8202,train,0.509456,-0.097929
8203,train,0.513062,-0.097929
8204,train,0.319099,0.464463
8205,train,0.310533,0.464463
8206,train,0.711698,0.543609
...,...,...,...
1595,test,0.505724,0.175335
2335,test,0.676842,0.317968
5274,test,0.307629,0.278606
6464,test,0.468892,0.660134


### No transformation

In [8]:
# tmp
df_tmp = df.copy()

# data split
df_train = df_tmp[df_tmp['data_set'] == 'train'].copy()
df_valid = df_tmp[df_tmp['data_set'] == 'valid'].copy()
df_test = df_tmp[df_tmp['data_set'] == 'test'].copy()

# list cols in model
list_cols = [
    'pd',
]
# init
cls_model = LinearRegression()
# fit
cls_model.fit(
    df_train[list_cols], 
    df_train['target'],
)
# predict
y_hat_train = cls_model.predict(df_train[list_cols])
y_hat_valid = cls_model.predict(df_valid[list_cols])
y_hat_test = cls_model.predict(df_test[list_cols])
# get scores
flt_score_train = np.sqrt(skm.mean_squared_error(
    y_true=df_train['target'], 
    y_pred=y_hat_train,
))
flt_score_valid = np.sqrt(skm.mean_squared_error(
    y_true=df_valid['target'], 
    y_pred=y_hat_valid,
))
flt_score_test = np.sqrt(skm.mean_squared_error(
    y_true=df_test['target'], 
    y_pred=y_hat_test,
))
print(f'Train: {flt_score_train:0.4f}')
print(f'Valid: {flt_score_valid:0.4f}')
print(f'Test: {flt_score_test:0.4f}')

Train: 0.2667
Valid: 0.2805
Test: 0.2857


### Log

In [9]:
# tmp
df_tmp = df.copy()

# transform
df_tmp['pd'] = np.log(df_tmp['pd'])

# data split
df_train = df_tmp[df_tmp['data_set'] == 'train'].copy()
df_valid = df_tmp[df_tmp['data_set'] == 'valid'].copy()
df_test = df_tmp[df_tmp['data_set'] == 'test'].copy()

# list cols in model
list_cols = [
    'pd',
]
# init
cls_model = LinearRegression()
# fit
cls_model.fit(
    df_train[list_cols], 
    df_train['target'],
)
# predict
y_hat_train = cls_model.predict(df_train[list_cols])
y_hat_valid = cls_model.predict(df_valid[list_cols])
y_hat_test = cls_model.predict(df_test[list_cols])
# get scores
flt_score_train = np.sqrt(skm.mean_squared_error(
    y_true=df_train['target'], 
    y_pred=y_hat_train,
))
flt_score_valid = np.sqrt(skm.mean_squared_error(
    y_true=df_valid['target'], 
    y_pred=y_hat_valid,
))
flt_score_test = np.sqrt(skm.mean_squared_error(
    y_true=df_test['target'], 
    y_pred=y_hat_test,
))
print(f'Train: {flt_score_train:0.4f}')
print(f'Valid: {flt_score_valid:0.4f}')
print(f'Test: {flt_score_test:0.4f}')

Train: 0.2670
Valid: 0.2807
Test: 0.2855


### Log10

In [10]:
# tmp
df_tmp = df.copy()

# transform
df_tmp['pd'] = np.log10(df_tmp['pd'])

# data split
df_train = df_tmp[df_tmp['data_set'] == 'train'].copy()
df_valid = df_tmp[df_tmp['data_set'] == 'valid'].copy()
df_test = df_tmp[df_tmp['data_set'] == 'test'].copy()

# list cols in model
list_cols = [
    'pd',
]
# init
cls_model = LinearRegression()
# fit
cls_model.fit(
    df_train[list_cols], 
    df_train['target'],
)
# predict
y_hat_train = cls_model.predict(df_train[list_cols])
y_hat_valid = cls_model.predict(df_valid[list_cols])
y_hat_test = cls_model.predict(df_test[list_cols])
# get scores
flt_score_train = np.sqrt(skm.mean_squared_error(
    y_true=df_train['target'], 
    y_pred=y_hat_train,
))
flt_score_valid = np.sqrt(skm.mean_squared_error(
    y_true=df_valid['target'], 
    y_pred=y_hat_valid,
))
flt_score_test = np.sqrt(skm.mean_squared_error(
    y_true=df_test['target'], 
    y_pred=y_hat_test,
))
print(f'Train: {flt_score_train:0.4f}')
print(f'Valid: {flt_score_valid:0.4f}')
print(f'Test: {flt_score_test:0.4f}')

Train: 0.2670
Valid: 0.2807
Test: 0.2855


### Square

In [11]:
# tmp
df_tmp = df.copy()

# transform
df_tmp['pd'] = df_tmp['pd'] ** 2

# data split
df_train = df_tmp[df_tmp['data_set'] == 'train'].copy()
df_valid = df_tmp[df_tmp['data_set'] == 'valid'].copy()
df_test = df_tmp[df_tmp['data_set'] == 'test'].copy()

# list cols in model
list_cols = [
    'pd',
]
# init
cls_model = LinearRegression()
# fit
cls_model.fit(
    df_train[list_cols], 
    df_train['target'],
)
# predict
y_hat_train = cls_model.predict(df_train[list_cols])
y_hat_valid = cls_model.predict(df_valid[list_cols])
y_hat_test = cls_model.predict(df_test[list_cols])
# get scores
flt_score_train = np.sqrt(skm.mean_squared_error(
    y_true=df_train['target'], 
    y_pred=y_hat_train,
))
flt_score_valid = np.sqrt(skm.mean_squared_error(
    y_true=df_valid['target'], 
    y_pred=y_hat_valid,
))
flt_score_test = np.sqrt(skm.mean_squared_error(
    y_true=df_test['target'], 
    y_pred=y_hat_test,
))
print(f'Train: {flt_score_train:0.4f}')
print(f'Valid: {flt_score_valid:0.4f}')
print(f'Test: {flt_score_test:0.4f}')

Train: 0.2671
Valid: 0.2811
Test: 0.2853


### Square root

In [12]:
# tmp
df_tmp = df.copy()

# transform
df_tmp['pd'] = np.sqrt(df_tmp['pd'])

# data split
df_train = df_tmp[df_tmp['data_set'] == 'train'].copy()
df_valid = df_tmp[df_tmp['data_set'] == 'valid'].copy()
df_test = df_tmp[df_tmp['data_set'] == 'test'].copy()

# list cols in model
list_cols = [
    'pd',
]
# init
cls_model = LinearRegression()
# fit
cls_model.fit(
    df_train[list_cols], 
    df_train['target'],
)
# predict
y_hat_train = cls_model.predict(df_train[list_cols])
y_hat_valid = cls_model.predict(df_valid[list_cols])
y_hat_test = cls_model.predict(df_test[list_cols])
# get scores
flt_score_train = np.sqrt(skm.mean_squared_error(
    y_true=df_train['target'], 
    y_pred=y_hat_train,
))
flt_score_valid = np.sqrt(skm.mean_squared_error(
    y_true=df_valid['target'], 
    y_pred=y_hat_valid,
))
flt_score_test = np.sqrt(skm.mean_squared_error(
    y_true=df_test['target'], 
    y_pred=y_hat_test,
))
print(f'Train: {flt_score_train:0.4f}')
print(f'Valid: {flt_score_valid:0.4f}')
print(f'Test: {flt_score_test:0.4f}')

Train: 0.2667
Valid: 0.2804
Test: 0.2857


### Cube

In [13]:
# tmp
df_tmp = df.copy()

# transform
df_tmp['pd'] = df_tmp['pd'] ** 3

# data split
df_train = df_tmp[df_tmp['data_set'] == 'train'].copy()
df_valid = df_tmp[df_tmp['data_set'] == 'valid'].copy()
df_test = df_tmp[df_tmp['data_set'] == 'test'].copy()

# list cols in model
list_cols = [
    'pd',
]
# init
cls_model = LinearRegression()
# fit
cls_model.fit(
    df_train[list_cols], 
    df_train['target'],
)
# predict
y_hat_train = cls_model.predict(df_train[list_cols])
y_hat_valid = cls_model.predict(df_valid[list_cols])
y_hat_test = cls_model.predict(df_test[list_cols])
# get scores
flt_score_train = np.sqrt(skm.mean_squared_error(
    y_true=df_train['target'], 
    y_pred=y_hat_train,
))
flt_score_valid = np.sqrt(skm.mean_squared_error(
    y_true=df_valid['target'], 
    y_pred=y_hat_valid,
))
flt_score_test = np.sqrt(skm.mean_squared_error(
    y_true=df_test['target'], 
    y_pred=y_hat_test,
))
print(f'Train: {flt_score_train:0.4f}')
print(f'Valid: {flt_score_valid:0.4f}')
print(f'Test: {flt_score_test:0.4f}')

Train: 0.2675
Valid: 0.2819
Test: 0.2849


### Cube root

In [14]:
# tmp
df_tmp = df.copy()

# transform
df_tmp['pd'] = np.cbrt(df_tmp['pd'])

# data split
df_train = df_tmp[df_tmp['data_set'] == 'train'].copy()
df_valid = df_tmp[df_tmp['data_set'] == 'valid'].copy()
df_test = df_tmp[df_tmp['data_set'] == 'test'].copy()

# list cols in model
list_cols = [
    'pd',
]
# init
cls_model = LinearRegression()
# fit
cls_model.fit(
    df_train[list_cols], 
    df_train['target'],
)
# predict
y_hat_train = cls_model.predict(df_train[list_cols])
y_hat_valid = cls_model.predict(df_valid[list_cols])
y_hat_test = cls_model.predict(df_test[list_cols])
# get scores
flt_score_train = np.sqrt(skm.mean_squared_error(
    y_true=df_train['target'], 
    y_pred=y_hat_train,
))
flt_score_valid = np.sqrt(skm.mean_squared_error(
    y_true=df_valid['target'], 
    y_pred=y_hat_valid,
))
flt_score_test = np.sqrt(skm.mean_squared_error(
    y_true=df_test['target'], 
    y_pred=y_hat_test,
))
print(f'Train: {flt_score_train:0.4f}')
print(f'Valid: {flt_score_valid:0.4f}')
print(f'Test: {flt_score_test:0.4f}')

Train: 0.2668
Valid: 0.2805
Test: 0.2857


### Reciprocal

In [15]:
# tmp
df_tmp = df.copy()

# transform
df_tmp['pd'] = 1 / df_tmp['pd']

# data split
df_train = df_tmp[df_tmp['data_set'] == 'train'].copy()
df_valid = df_tmp[df_tmp['data_set'] == 'valid'].copy()
df_test = df_tmp[df_tmp['data_set'] == 'test'].copy()

# list cols in model
list_cols = [
    'pd',
]
# init
cls_model = LinearRegression()
# fit
cls_model.fit(
    df_train[list_cols], 
    df_train['target'],
)
# predict
y_hat_train = cls_model.predict(df_train[list_cols])
y_hat_valid = cls_model.predict(df_valid[list_cols])
y_hat_test = cls_model.predict(df_test[list_cols])
# get scores
flt_score_train = np.sqrt(skm.mean_squared_error(
    y_true=df_train['target'], 
    y_pred=y_hat_train,
))
flt_score_valid = np.sqrt(skm.mean_squared_error(
    y_true=df_valid['target'], 
    y_pred=y_hat_valid,
))
flt_score_test = np.sqrt(skm.mean_squared_error(
    y_true=df_test['target'], 
    y_pred=y_hat_test,
))
print(f'Train: {flt_score_train:0.4f}')
print(f'Valid: {flt_score_valid:0.4f}')
print(f'Test: {flt_score_test:0.4f}')

Train: 0.2688
Valid: 0.2839
Test: 0.2844
